# Demo and synthetic characterisation

**Part A (≈5 min):** a live demo — invert frozen TRIBE v2 on a known representation and watch recovery converge.

**Part B (hours on a T4):** every synthetic experiment in the manuscript (Sections IV-A to IV-E). Each sub-command writes `results/synth_<name>.json`.

No Llama access is needed: the inversion starts after TRIBE v2's feature extractors. Use `Runtime → Change runtime type → T4 GPU`.

## Step 1 · Install and clone (then Runtime → Restart session)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install uv
!uv pip -q install --system "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"
!uv pip -q install --system "transformers<4.57" "torchaudio==2.6.0" "numpy<2.1" h5py scikit-learn
!git clone -q https://github.com/Ivxn404/semantic-decoding-tribe-inversion /content/repo || (cd /content/repo && git pull -q)
print('INSTALL DONE - now click Runtime > Restart session, then continue with the next step')

## Step 2 · Settings and weights

In [ ]:
import os, sys
os.environ['PROJECT_ROOT'] = '/content/repo'
os.environ['TRIBE_WEIGHTS'] = '/content/repo/weights'
os.chdir('/content/repo/scripts'); sys.path.insert(0, '/content/repo/scripts')
sys.path = [p for p in sys.path if not p.endswith('tribev2')]

In [ ]:
from huggingface_hub import hf_hub_download
for f in ['config.yaml', 'best.ckpt']:
    hf_hub_download('facebook/tribev2', f, local_dir='/content/repo/weights')
!ls -la /content/repo/weights

## Part A · Live demo: invert TRIBE v2 on a known representation

In [ ]:
import torch, matplotlib.pyplot as plt
from load_tribe import load, make_fwd
from synthetic_characterisation import make_truth, metrics
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
net, _ = load(); net = net.to(dev); fwd = make_fwd(net)
x_true = make_truth().to(dev)                       # known 32 x 1152 representation
y = fwd(x_true).detach()                            # predicted fMRI, 20 484 vertices x 100 steps
g = torch.Generator().manual_seed(999)
x = (0.01 * torch.randn(x_true.shape, generator=g)).to(dev).requires_grad_(True)
STEPS = 3000
opt = torch.optim.Adam([x], lr=5e-2); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STEPS)
log = []
for i in range(STEPS + 1):
    if i % 100 == 0:
        m = metrics(fwd, x.detach(), x_true); log.append((i, m['cosine'], m['enc_rel_error']))
        print(f"step {i:5d}  recovery cosine {m['cosine']:.3f}  encoding error {m['enc_rel_error']:.2e}")
    if i == STEPS: break
    opt.zero_grad(); ((fwd(x) - y) ** 2).mean().backward(); opt.step(); sch.step()
s, c, e = zip(*log)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(s, c); ax[0].set_title('Recovery cosine to ground truth'); ax[0].set_xlabel('step')
ax[1].semilogy(s, e); ax[1].set_title('Encoding error (relative)'); ax[1].set_xlabel('step')
plt.show()

## Part B · Full synthetic characterisation
Run one line at a time, or `all`. Use `--quick` first to check everything runs (≈5 min).

In [ ]:
!python synthetic_characterisation.py all --quick 2>&1 | grep -v -i warn | tail -5

In [ ]:
!python synthetic_characterisation.py dissociation 2>&1 | grep -v -i warn
!python synthetic_characterisation.py inits barrier sensitivity 2>&1 | grep -v -i warn
!python synthetic_characterisation.py gain noise extent priors ridge bandwidth manifold 2>&1 | grep -v -i warn

## Part C · Regional analysis (Section IV-F)

In [ ]:
os.environ['ATLAS_DIR'] = '/content/atlas'
!python roi_sensitivity.py && python roi_share_stats.py && python roi_inversion.py && python roi_dimensionality.py